# Machine Learning: 2 Models MNISTBaselineNet vs. MNISTDropoutNet

* MNISTBaselineNet: ohne Dropout, ohne Early Stopping
* MNISTDropoutNet: mit Dropout und Early Stopping über val_loss

In [7]:
%load_ext autoreload
%autoreload 2

import mlkit as mlk

import torch
from torch import nn # The neural-network building blocks, such as Linear and ReLU.

from torchvision import datasets, transforms # Provides MNIST and image preprocessing.
from torch.utils.data import DataLoader, random_split # Provides data loading and splitting.


## Functions

Downloading MNIST-Dataset

In [ ]:
def dataloader(batch_size=BATCH_SIZE):
    """
    Download MNIST and wrap the train and test splits in DataLoaders.
    """

    # Normalisierung/Skalierung
    # Pixelwerte von 0 bis 255 auf in PyTorch-Tensors mit float32-Werte im Bereich von [0, 1]
    transform = transforms.ToTensor() # ToTensor transformiert MNIST-Bilder von 0 bis 255 in PyTorch-Tensors mit float32-Werte im Bereich von [0, 1]

    # Training- und Test-Datensätze laden
    full_train_dataset = datasets.MNIST(
        root="data_mnist", # Speicherort
        train=True,
        download=True,
        transform=transform # ToTensor() anwenden
    )

    test_dataset = datasets.MNIST(
        root="data_mnist",
        train=False,
        download=True,
        transform=transform
    )

    # Split train_dataset into train_dataset and val_dataset
    # Why val_dataset?
    # - das Modell während des Trainings zu überprüfen
    # - Overfitting zu erkennen
    # - Hyperparameter zu testen
    train_dataset, val_dataset = random_split( # random_split() teilt den Datensatz in zwei unabhängige Datensätze
        full_train_dataset,
        [50_000, 10_000], # [train_size, val_size]
        generator=torch.Generator().manual_seed(42)
    )

    # Erstellen von DataLoadern
    # DataLoader() erstellt mini-batches aus dem Datensatz
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # shuffle=True: Shuffle training data so batches vary across epochs
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False) # shuffle=False: Validation soll stabil und reproduzierbar sein; Keep test order stable because evaluation does not learn; 
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) # shuffle=False: Keep test order stable because evaluation does not learn

    return train_loader, val_loader, test_loader

## 1. Basics

### 1.1 Setup

In [9]:
# batch size of 32 performs best cf. REVISITING SMALL BATCH TRAINING FOR DEEP NEURAL NETWORKS
BATCH_SIZE = 32  # The model sees 64 images before each weight update.

In [8]:
# Set device
device = mlk.choose_device()
print(f"Using device: {device}")

Using device: mps


### 1.2 Loading data

In [14]:
train_loader, val_loader, test_loader = dataloader()

100%|██████████| 9.91M/9.91M [00:05<00:00, 1.96MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 240kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.63MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.4MB/s]


# 2. MNISTBaselineNet

* MNISTBaselineNet: ohne Dropout, ohne Early Stopping

Definitionen
* **Logits**: rohe Ausgabewerte eines neuronalen Netzes vor der Umwandlung in Wahrscheinlichkeiten

In [ ]:
class MNISTBaselineNet(nn.Module):
    """
    Einfaches Modell ohne Dropout.
    """

    def __int__(self): # Initialisierung bzw.  wird beim Erstellen des Modells ausgeführt
        super().__init__() # initialisiert die Elternklasse (nn.Module)
        self.network = nn.Sequential( # Schichten werden nacheinander durchlaufen; Der Output einer Schicht ist der Input der nächsten
            nn.Flatten(), # MNIST-Bilder werden "geflattened" von 28x28 in einen Vektor mit 784 Pixeln/Werten
            nn.Linear(28 * 28, 128), # erste Verdichtung der Informationen; Input: 784 Features/Pixel; Output: 128 Neuronen 
            nn.ReLU(), # Non-linear activation function, d.h. fügt Nicht-Lineraität hinzu, damit das Netz komplexe Muster lernen kann
            nn.Linear(128, 64), # Reduziert die Daten von 128 auf 64 Neuronen -> Netz lernt abstrakte Merkmale
            nn.ReLU(),
            nn.Linear(64, 10) # Output: 10 Klassen= Ziffern 0-9; Outputs sind: Logits (keine Wahrscheinlichkeiten!)
        )
        